In [ ]:

import sqlalchemy as db
from sqlalchemy import text
import conn

engine = db.create_engine(conn.get_connection())

queries = [
    """
    CREATE TABLE IF NOT EXISTS elpris_dagligt (
        datum DATE,
        elomrade VARCHAR(10),
        genomsnitt_sek_kwh FLOAT
    )
    """,

    """
    CREATE TABLE IF NOT EXISTS fordon_i_trafik (
        id INTEGER PRIMARY KEY
    )
    """,

    """
    CREATE TABLE IF NOT EXISTS nyregistrerade_personbilar (
        id INTEGER PRIMARY KEY
    )
    """
]

with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))

In [67]:
import requests
import time
from datetime import date, timedelta

import sqlalchemy as db
from sqlalchemy import text

import conn

START_DATE = date(2022, 1, 1)
END_DATE = date(2025, 12, 31)
PRICE_AREA = "SE3"

current_date = START_DATE
rows = []

while current_date <= END_DATE:
    year = current_date.strftime("%Y")
    month_day = current_date.strftime("%m-%d")

    url = (
        f"https://www.elprisetjustnu.se/api/v1/prices/"
        f"{year}/{month_day}_{PRICE_AREA}.json"
    )

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:
            data = response.json()

            prices = [
                float(row["SEK_per_kWh"])
                for row in data
                if row.get("SEK_per_kWh") is not None
            ]

            if prices:
                daily_average = sum(prices) / len(prices)
                rows.append({
                    "datum": current_date.isoformat(),
                    "elomrade": PRICE_AREA,
                    "genomsnitt_sek_kwh": daily_average
                })

        elif response.status_code == 404:
            print(f"{current_date}: Ingen data")

        else:
            print(f"{current_date}: HTTP {response.status_code}")

    except requests.exceptions.RequestException as e:
        print(f"{current_date}: FEL - {e}")

    current_date += timedelta(days=1)
    time.sleep(0.1)


engine = db.create_engine(conn.get_connection())


# with engine.begin() as connection:
#     connection.execute(text("""
#         CREATE TABLE IF NOT EXISTS elpris_dagligt (
#             datum DATE,
#             elomrade VARCHAR(10),
#             genomsnitt_sek_kwh FLOAT
#         )
#     """))

with engine.begin() as connection:

    connection.execute(
        text("""
            INSERT INTO elpris_dagligt
            (datum, elomrade, genomsnitt_sek_kwh)
            VALUES (
                :datum,
                :elomrade,
                :genomsnitt_sek_kwh
            )
        """),
        rows
    )


In [ ]:
# import sqlalchemy as db
# from sqlalchemy import text

# import conn



# engine = db.create_engine(conn.get_connection())
# queries = [
#     """
#     drop TABLE elpris_dagligt 
#     """,

#     """
#     drop TABLE fordon_i_trafik 
#     """,

#     """
#     drop TABLE nyregistrerade_personbilar 
#     """
# ]

# with engine.begin() as connection:
#     for query in queries:
#         connection.execute(text(query))